# EchoTwin — real-corpus LoRA style transfer (Kaggle GPU)

This notebook runs the **actual EchoTwin pipeline** on a **real, public,
multi-conversation corpus** and reports measured numbers:

1. install EchoTwin from a private Kaggle dataset (no secrets in this file),
2. fetch a real chat corpus and convert it to WhatsApp-style exports,
3. import it with the production parser, pick the author, build the dataset,
   split by conversation (leakage-resistant),
4. train a LoRA adapter (SFT with verified target masking),
5. evaluate **base / style-profile / style-rag / LoRA** on the *same* held-out
   prompts, and report style similarity, prompt responsiveness, contamination
   and failures.

**Honesty note:** every number below was produced by running the code in this
notebook. The corpus author is a *role* in a public corpus, not one individual,
so the result shows that the method captures a consistent speaker register -
it is not a claim about any specific person.


In [ ]:
import importlib.metadata as _md
# Pin the already-imported numpy/pandas: upgrading them under a live kernel breaks the ABI.
open("/tmp/pins.txt", "w").write("\n".join(f"{n}=={_md.version(n)}" for n in ("numpy", "pandas")))
!pip install -q -c /tmp/pins.txt --upgrade "transformers>=5.0" "peft>=0.21" "trl>=0.15" \
    accelerate datasets huggingface_hub pyarrow soundfile scipy pandas pytest
!pip uninstall -y -q torchao  # preinstalled 0.10 is rejected by peft; unused here
import torch, transformers, peft
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("transformers", transformers.__version__, "peft", peft.__version__)


In [ ]:
# Install EchoTwin from the PRIVATE dataset mirror (no credential in this file)
!mkdir -p /kaggle/working/src
!kaggle datasets download -d shashwat1729/echotwin-source -p /kaggle/working/src --unzip --force
!cd /kaggle/working/src && for z in *.zip; do [ -f "$z" ] && unzip -o "$z"; done; ls /kaggle/working/src
!pip install -q --ignore-requires-python -c /tmp/pins.txt -e "/kaggle/working/src[voice,train,lang,dev]"
import sys; sys.path.insert(0, "/kaggle/working/src/src")  # .pth files are read at startup only
import echotwin
print("echotwin", echotwin.__version__, "from", echotwin.__file__)


In [ ]:
import glob, os, shutil
os.makedirs("/kaggle/working/personal_chats", exist_ok=True)
files = glob.glob("/kaggle/input/echotwin-personal/chats/*_chat.txt")
for f in files:
    shutil.copy(f, "/kaggle/working/personal_chats/")
print("chats:", len(files))
print(sorted(os.listdir("/kaggle/input/echotwin-personal/voice"))[:3])


In [ ]:
# 2) Run the experiment: dataset -> conversation split -> LoRA -> evaluation.
#    Every method sees the SAME held-out prompts. (T4 GPU, your private chats; adapter + voice samples saved to /kaggle/working)
!cd /kaggle/working && python src/scripts/run_style_experiment.py \
    --corpus personal_chats --author "Me" \
    --out artifacts/experiments/kaggle \
    --model qwen3-1.7b \
    --methods base,style_profile,style_rag,ngram,lora,style_rag_ft \
    --train-lora --epochs 2 --lora-rank 16 \
    --learning-rate 1e-4 \
    --max-train-examples 2000 \
    --max-eval-prompts 100 --max-new-tokens 56


In [ ]:
# 3) Show the report and the generated replies
import json, pathlib
report = json.loads(pathlib.Path("/kaggle/working/artifacts/experiments/kaggle/report.json").read_text())
print("corpus:", report["corpus"])
print("splits:", report["splits"])
for m in report["methods"]:
    s = m.get("style") or {}
    print(f"{m['method']:<14} cos3={s.get('char_ngram_cosine_3')} "
          f"len_ratio={s.get('length_mean_ratio')} "
          f"distinct={(m.get('prompt_discrimination') or {}).get('distinct_ratio')} "
          f"uptake={(m.get('prompt_uptake') or {}).get('mean_uptake')} "
          f"contam={(m.get('contamination') or {}).get('verdict')} failed={m['n_failed']}")
print()
for name in ("plain", "style_rag", "lora"):
    p = pathlib.Path(f"/kaggle/working/artifacts/experiments/kaggle/predictions_{name}.jsonl")
    if not p.exists():
        continue
    print(f"--- {name} ---")
    for line in p.read_text(encoding="utf-8").splitlines()[:3]:
        r = json.loads(line)
        print("  prompt:", (r.get("context") or [""])[-1][:70] if r.get("context") else "")
        print("  model :", (r.get("generated") or "")[:100])
        print("  human :", (r.get("reference") or "")[:100])


In [ ]:
# 5) VOICE CLONING (VoxCPM). Runs only if voice consent was recorded in the
#    dataset (granted by the speaker in the EchoTwin studio) - otherwise skipped.
import json, pathlib, shutil, subprocess, sys
import glob
_hits = glob.glob("/kaggle/input/**/voice/manifest.json", recursive=True)
vin = pathlib.Path(_hits[0]).parent.parent if _hits else pathlib.Path("/kaggle/input/echotwin-personal")
consent = vin / "voice_consent.json"
if not consent.exists():
    print("SKIPPED: no voice_consent.json in the dataset - voice consent has not been granted.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python", "voxcpm", "soundfile"], check=True)
    from voxcpm import VoxCPM
    import soundfile as sf
    manifest = json.loads((vin / "voice" / "manifest.json").read_text(encoding="utf-8"))
    # reference: the clip closest to 12 s that has a script-aligned transcript
    ref = min((m for m in manifest if m["transcript"]), key=lambda m: abs(m["seconds"] - 12))
    ref_path = str(vin / "voice" / ref["file"])
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    out = pathlib.Path("/kaggle/working/voice_out"); out.mkdir(exist_ok=True)
    tests = {
        "hinglish": "Yaar kal milte hain, main ek naya idea lekar aa raha hoon, tum bas time pe aa jaana.",
        "english": "This is a short test of my cloned voice, speaking in a calm and natural way.",
        "hindi": "नमस्ते, यह मेरी आवाज़ का एक छोटा सा परीक्षण है।",
    }
    for name, text in tests.items():
        for mode in ("reference", "continuation"):
            kw = {"text": text, "reference_wav_path": ref_path}
            if mode == "continuation":
                kw.update(prompt_wav_path=ref_path, prompt_text=ref["transcript"])
            wav = model.generate(**kw)
            sf.write(str(out / f"{name}_{mode}.wav"), wav, model.tts_model.sample_rate)
            print("wrote", f"{name}_{mode}.wav")
    shutil.copy(ref_path, out / "reference_used.wav")


In [ ]:
# 4) Persist the results as a Kaggle dataset so they can be pulled back
#    into the repository and cited in the docs.
!cd /kaggle/working && mkdir -p echotwin-results && cp -r artifacts/experiments/kaggle/* echotwin-results/ 2>/dev/null || true
!ls -la /kaggle/working/echotwin-results | head


## Interpreting the numbers

* **cos3** - character 3-gram cosine between the model's replies and the
  author's real held-out replies. Higher = closer surface style.
* **len_ratio** - mean generated length / mean author length. ~1.0 means the
  model matches the author's message length.
* **distinct** - share of *different* replies across prompts. A low value means
  the model ignores the prompt and repeats a canned answer.
* **uptake** - share of the prompt's content words echoed in the reply
  (a responsiveness proxy, not a correctness measure).
* **contam** - memorization verdict: `clean` / `some_overlap` / `high_overlap`.
* **failed** - generations that errored. A method with failures is not
  comparable on the other columns.

Caveats: the corpus is public dialogue (not WhatsApp, not Hinglish), the
author is a role rather than one person, and a single seed per prompt means
small differences are noise. The report JSON records all of this.
